# Iceberg layer review -- silver and gold

Read-only look at the **Iceberg** layers, through **Spark**.

Why not pandas, the way `review_layers.ipynb` reads raw and quarantine? An
Iceberg table is a set of parquet files *plus metadata deciding which of them
are live*. Both `merge` and `delete_insert` rewrite files, so reading the
parquet directly would show rows a later snapshot deleted -- silently, with no
hint that the numbers are wrong. Spark is the authoritative reader.

The helpers live in `review_gold.py` beside this notebook and build every
identifier from `config/local.yaml`, so this cannot drift from where the jobs
actually write.

**The session costs ~30s to start** (JVM, then Ivy resolving the Iceberg
jars). The kernel holds it across every cell, so you pay that once -- which is
the whole reason this is a notebook and not a script.

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "..")   # the python_scripts package, from the repo root
sys.path.insert(0, ".")    # review_gold.py, beside this notebook

import review_gold as rg

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

spark, cfg = rg.connect("prod")      # ~30s on a cold kernel
print(spark.conf.get("spark.sql.catalog.iceberg.warehouse"))
print(f"gold tables in config: {sorted(cfg.gold)}")

s3://globalpartners-project-cjk/prod/warehouse
gold tables in config: ['daily_restaurant_sales', 'restaurant_revenue_rank']


## 1. What exists

Start here after any run. Row counts come from the Iceberg **manifests**, not
from scanning data, so this stays cheap however large the tables get.

A table listed with `rows = 0` means a job created it and wrote an empty
frame -- a different failure from the job never running, and worth telling
apart.

If `gold` comes back empty, no gold job has run against *this* warehouse yet.

In [2]:
rg.namespaces(spark)

,namespace
0,gold
1,silver


In [3]:
rg.inventory(spark, cfg, "silver")

,table,rows,files,bytes,snapshots,last_write
0,date_dim,365,1,3569,4,2026-10-23 10:39:41.789
1,order_item_options,193017,1,3322345,3,2026-10-23 10:40:15.079
2,order_items,203518,47,7111292,2,2026-10-23 10:40:05.451


In [4]:
rg.inventory(spark, cfg, "gold")

,table,rows,files,bytes,snapshots,last_write
0,daily_restaurant_sales,13413,47,294610,1,2026-10-23 12:35:57.562
1,restaurant_revenue_rank,28,1,3735,1,2026-10-23 12:36:03.882


## 2. The gold tables

`daily_restaurant_sales` is one row per `sales_date` x `restaurant_id`, built
incrementally -- each run recomputes only the event dates the batch touched.

`restaurant_revenue_rank` is one row per restaurant over all history, rebuilt
in full every run. It has to be separate: a rank over all history cannot be
computed from an incremental slice, and ranks cannot be merged.

In [5]:
rg.schema(spark, cfg, "gold", "daily_restaurant_sales")

,col_name,data_type,comment
0,sales_date,date,NaN
1,restaurant_id,string,NaN
2,total_orders,bigint,NaN
3,line_items,bigint,NaN
4,units,bigint,NaN
5,total_revenue,"decimal(18,2)",NaN
6,avg_order_total,"decimal(10,2)",NaN
7,week_start,date,NaN
8,batch_date,date,NaN
9,updated_at,timestamp,NaN


In [9]:
rg.peek(spark, cfg, "gold", "restaurant_revenue_rank",
        limit=15, order_by="revenue_rank")

,restaurant_id,total_orders,line_items,units,total_revenue,avg_order_total,revenue_rank,batch_date,updated_at
0,5fece84aadb0d51509c36f22,20727,32459,36038,295715.38,14.27,1,2026-10-16,2026-10-23 12:36:00.078967
1,6054db3495b70198148b456f,14183,23135,25095,186469.57,13.15,2,2026-10-16,2026-10-23 12:36:00.078967
2,5e7e35ec902ad5ac017b242a,13525,21416,23277,183593.09,13.57,3,2026-10-16,2026-10-23 12:36:00.078967
3,5f6a6c1537ab46bd38e9df75,12207,18888,20978,163702.12,13.41,4,2026-10-16,2026-10-23 12:36:00.078967
4,5edff313902ad5580e9c3f30,8017,10569,14406,127510.54,15.91,5,2026-10-16,2026-10-23 12:36:00.078967
5,62f2ce9824813746ce6f5140,8303,12571,13575,113872.75,13.71,6,2026-10-16,2026-10-23 12:36:00.078967
6,622107b40ac81503e0369ca6,6529,11594,12637,101378.59,15.53,7,2026-10-16,2026-10-23 12:36:00.078967
7,6054db3395b70198148b456e,6932,11776,12659,96101.46,13.86,8,2026-10-16,2026-10-23 12:36:00.078967
8,63bc98a7519adc105105a990,6396,10313,11001,92017.14,14.39,9,2026-10-16,2026-10-23 12:36:00.078967
9,6351b7d1a036663f05239b60,5797,8897,9659,75836.94,13.08,10,2026-10-16,2026-10-23 12:36:00.078967


In [ ]:
rg.leaderboard(spark, cfg)

## 3. The measures the dashboard needs

Total revenue, average order value, orders per day, orders per week.

Per day comes straight off the table. Per week is a rollup on `week_start`,
and the one thing to get right is that **`avg_order_total` is recomputed, not
averaged** -- a mean of means weights a 2-order Tuesday like a 200-order
Saturday. `rg.weekly()` does it as `sum(revenue) / sum(orders)`.

`sum(total_orders)` *is* safe, which is less obvious: it is a distinct count
within a day, and an order has exactly one creation timestamp, so the daily
sets are disjoint and cannot double-count.

In [ ]:
rg.weekly(spark, cfg).head(20)

## 4. How the table was written

`snapshots` is one row per write, and its `operation` column is the useful
part: `append`, `overwrite` or `delete`. This is how you confirm a
`delete_insert` actually deleted, or that a `replace` rewrote rather than
appended. The `summary` map carries added and deleted record and file counts.

In [ ]:
rg.snapshots(spark, cfg, "gold", "daily_restaurant_sales")

`partitions` answers the small-file question the Readme raises: rows and files
per partition. `months(sales_date)` should give partitions worth having rather
than hundreds holding a few KB each.

In [ ]:
rg.partitions(spark, cfg, "gold", "daily_restaurant_sales")

In [ ]:
rg.files(spark, cfg, "gold", "daily_restaurant_sales", limit=20)

## 5. Time travel

Every write leaves the previous snapshot readable. Take an id from
`snapshots()` above and compare -- the honest way to check what a transform
change actually did to the numbers, rather than reasoning about it.

In [ ]:
# Pick the two most recent snapshots and diff the row counts.
snaps = rg.snapshots(spark, cfg, "gold", "daily_restaurant_sales")

if len(snaps) >= 2:
    before, after = snaps.snapshot_id.iloc[-2], snaps.snapshot_id.iloc[-1]
    b = rg.as_of(spark, cfg, "gold", "daily_restaurant_sales", before)
    a = rg.as_of(spark, cfg, "gold", "daily_restaurant_sales", after)
    print(f"snapshot {before}: {b.count():,} rows")
    print(f"snapshot {after}: {a.count():,} rows")
else:
    print("need at least two snapshots to diff")

## 6. Anything else

`rg.sql` runs arbitrary SQL with a row cap. Identifiers are three-part:
`iceberg.<layer>.<table>`, where the layer name comes from `layers:` in
config.

In [ ]:
rg.sql(spark, """
    SELECT sales_date, count(*) AS restaurants,
           sum(total_orders) AS orders,
           sum(total_revenue) AS revenue
    FROM iceberg.gold.daily_restaurant_sales
    GROUP BY sales_date
    ORDER BY sales_date DESC
""", limit=30)